In [1]:
import numpy as np

# Two states (s): FAR (0), NEAR (1)
# Two actions (a): FORWARD (0), ATTACK (1)
q = np.zeros((2, 2))
q[1, 1] = 1.0  # Assume attacking nearby is already known to be valuable

s, a, reward, next_s = 0, 0, 0.0, 1
alpha, gamma = 0.5, 0.9
q[s, a] += alpha * (reward + gamma * q[next_s].max() - q[s, a])
print(q)

[[0.45 0.  ]
 [0.   1.  ]]


**What happened?** FAR + FORWARD received an immediate reward of `0`, but it reached NEAR, where ATTACK has estimated value `1`. With `gamma=0.9` and `alpha=0.5`, the value of FORWARD increased from `0` to `0.45`.

A **Q-table** stores an estimated *future discounted return* for every `(state, action)` pair. It does **not** store the immediate reward or an action probability. We seeded one value above purely to make the update visible.

## FOOTSIES: learn a Q-table

In [2]:
import footsiesgym

rng = np.random.default_rng(42)
actions = [1, 2, 3]  # BACK, FORWARD, ATTACK
q = np.zeros((10, 3))  # 10 distance buckets, 3 choices
alpha, gamma, epsilon = 0.2, 0.95, 0.3

env = footsiesgym.make(
    platform="mac",
    launch_binaries=False,
    config={
        "port": 50051,
        "headless": False,
        "max_t": 250,
        "action_delay": 0,
        "guard_break_reward": 0.2,
        "use_reward_budget": True,
    },
)

def state(obs):
    return min(int(obs[0] * 10), 9)

returns = []
try:
    for episode in range(50):
        obs, _ = env.reset()
        total_reward = 0.0

        while True:
            s = state(obs["p1"])
            if rng.random() < epsilon:
                a = int(rng.integers(3))
            else:
                best = np.flatnonzero(q[s] == q[s].max())
                a = int(rng.choice(best))

            opponent_action = 2 if obs["p2"][0] > 0.2 else 3
            next_obs, rewards, terminated, truncated, _ = env.step({
                "p1": actions[a], "p2": opponent_action,
            })

            reward = float(rewards["p1"])
            done = terminated["p1"] or truncated["p1"]
            next_s = state(next_obs["p1"])
            target = reward if done else reward + gamma * q[next_s].max()
            q[s, a] += alpha * (target - q[s, a])

            total_reward += reward
            obs = next_obs
            if done:
                break

        returns.append(total_reward)
        if (episode + 1) % 10 == 0:
            print(f"Episode {episode + 1}: mean return = {np.mean(returns[-10:]):+.3f}")
finally:
    env.close()

Episode 10: mean return = +0.940
Episode 20: mean return = +0.860
Episode 30: mean return = +0.880
Episode 40: mean return = +0.720
Episode 50: mean return = +0.720


In [ ]:
names = ["BACK", "FORWARD", "ATTACK"]
for bucket, values in enumerate(q):
    best_action = names[int(np.argmax(values))]
    print(f"distance {bucket/10:.1f}–{(bucket+1)/10:.1f}: {best_action:<7} Q={values.round(2)}")

**Where does `rewards` come from?** `env.step(actions)` computes and returns it automatically according to the environment's rules. This notebook sets `guard_break_reward=0.2` and `use_reward_budget=True`, so the environment awards an intermediate reward when an opponent's guard health decreases and reduces the remaining win reward budget accordingly. Most steps still have reward `0`.

**What did we simplify?** Only distance is used as state; actions are BACK, FORWARD, ATTACK; the opponent follows a fixed rule. Action delay is disabled. This is a learning experiment, not a competitive agent. A higher training return is not guaranteed, and `max_t` can end a game before anyone wins.

**Core loop:** choose action (epsilon-greedy) → `env.step` → read reward → update one Q-table entry. Q-learning estimates action value using `reward + gamma * max(Q[next_state])`.